In [2]:
import os
import sys
import subprocess
import ctypes
import sys as sys_lib

PROJECT_ROOT = r'C:\Users\doman\Downloads\Project1-main\Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import thư viện cốt lõi của Manim
from manim import *

# 3. Import toàn bộ thư viện custom của bạn

from skills.fami_lib import *
from skills.fami_assets_helper import *
from skills.fami_effects import * # Import thêm file này phòng trường hợp các hiệu ứng nằm ở đây
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")
config.verbosity = "ERROR"
import warnings
warnings.filterwarnings('ignore')

c:\Users\doman\Downloads\Project1-main\manim_env\Lib\site-packages\manim_voiceover\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import sys
import os
from pathlib import Path
from IPython.display import Video, display
from scipy import stats
import math
import numpy as np
from manim import *
import random

# 1. Cấu hình đường dẫn và thông số render
config.media_dir = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\media"
config.pixel_width = 720   # Độ phân giải dọc 720x1280
config.pixel_height = 1280
config.frame_rate = 15
config.disable_caching = True 
config.verbosity = "ERROR" # Tắt các dòng log thừa

# 2. Đảm bảo thư mục tồn tại
os.makedirs(config.media_dir, exist_ok=True)

# 3. Ép môi trường nhận FFmpeg (nếu cần)
ffmpeg_bin = r"C:\ffmpeg\bin"
if ffmpeg_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + ffmpeg_bin

In [13]:
# Lưu ý: Đảm bảo FaMIBaseScene đã được định nghĩa trước đó hoặc đổi thành Scene
class mainScene(FaMIBaseScene):
    def construct(self):
        title = self.create_title("TÍNH TOÁN SỐ BÀN THẮNG KỲ VỌNG TRONG BÓNG ĐÁ")
        self.play(Write(title))
        # Subscene 1: Mô phỏng 1000 cú sút của cầu thủ
        # Dựng khung thành hoàn chỉnh trước (Kích thước to lấp đầy khoảng trống)
        goal_width = 5.0
        goal_height = 2.5
        y_ground = 1.5 

        post_bottom_left  = np.array([-goal_width/2, y_ground, 0])
        post_bottom_right = np.array([goal_width/2, y_ground, 0])
        goal_top_left     = np.array([-goal_width/2, y_ground + goal_height, 0])
        goal_top_right    = np.array([goal_width/2, y_ground + goal_height, 0])

        left_post  = Line(post_bottom_left, goal_top_left, color=WHITE, stroke_width=4)
        right_post = Line(post_bottom_right, goal_top_right, color=WHITE, stroke_width=4)
        crossbar   = Line(goal_top_left, goal_top_right, color=WHITE, stroke_width=4)
        ground_line = Line(np.array([-3.5, y_ground, 0]), np.array([3.5, y_ground, 0]), color=GRAY_D, stroke_width=2)
        complete_goal = VGroup(ground_line, left_post, right_post, crossbar)

        # Dòng chữ giả thuyết ở TRÊN khung thành
        suppose_text = Text("A player takes 1000 shots", font_size=20, color=WHITE)
        suppose_text.next_to(crossbar, UP, buff=0.95)

        self.play(Create(complete_goal), Write(suppose_text), run_time=1.5)
        self.wait(0.5)

        # Xuất hiện cầu thủ (Kích thước 2.5 đặt ở dưới)
        player_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\baycho.png"
        big_player = ImageMobject(player_img_path)
        big_player.set_width(2.5) 
        big_player.move_to([0, -2.5, 0]) 
        
        self.play(FadeIn(big_player, shift=UP))
        self.wait(0.5)

        # 3. Tạo hiệu ứng sút quả bóng nhiều lần vào gôn
        ball_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\ball.png"
        
        # Cho sút giả lập 4 quả đại diện (vào và trượt) để tạo hiệu ứng chuyển động nhanh
        shoot_targets = [
            np.array([-1.0, 2.5, 0]),  # Vào gôn góc trái
            np.array([1.5, 3.2, 0]),   # Trượt lên trời bên phải
            np.array([1.2, 2.0, 0]),   # Vào gôn góc phải
            np.array([-2.8, 1.8, 0]),  # Trượt ra biên trái
        ]

        for target in shoot_targets:
            # Khởi tạo bóng tại vị trí cầu thủ
            ball = ImageMobject(ball_img_path)
            ball.set_width(0.3) # Kích thước bóng phù hợp với cầu thủ
            ball.move_to(big_player.get_center() + UP * 0.5)
            
            # Hiện bóng, sút bay vào bia mục tiêu với tốc độ nhanh, rồi biến mất (để lại vệt dữ liệu)
            self.play(FadeIn(ball, scale=0.5), run_time=0.15)
            self.play(
                ball.animate.move_to(target).scale(0.7), 
                run_time=0.4, 
                rate_func=linear
            )
            self.play(FadeOut(ball), run_time=0.1)

        self.wait(0.5)

        # 4. Tạo hệ thống chấm (Thuật toán giới hạn tọa độ để né Title)
        num_dots = 200 
        goal_ratio = 0.27
        num_goals = int(num_dots * goal_ratio)
        num_misses = num_dots - num_goals

        all_dots = VGroup()
        
        # Chấm xanh (Goal): Nằm gọn bên trong khung gôn (An toàn hoàn toàn)
        for _ in range(num_goals):
            dot = Dot(
                point=[random.uniform(-goal_width/2 + 0.1, goal_width/2 - 0.1), 
                       random.uniform(y_ground + 0.1, y_ground + goal_height - 0.1), 0], 
                color=GREEN, radius=0.06
            )
            all_dots.add(dot)
            
        # Chấm đỏ (No Goal): Sút ra ngoài gôn hoặc vọt xà nhưng NÉ TIÊU ĐỀ
        for _ in range(num_misses):
            # Chọn ngẫu nhiên hướng trượt: Trượt trái, Trượt phải, hoặc Vọt xà
            x_miss = random.choice([
                random.uniform(-3.5, -goal_width/2), # Biên trái
                random.uniform(goal_width/2, 3.5),  # Biên phải
                random.uniform(-goal_width/2, goal_width/2) # Vùng giữa (vọt xà)
            ])
            
            if x_miss >= -goal_width/2 and x_miss <= goal_width/2:
                # Nếu ở vùng giữa gôn thì BẮT BUỘC phải vọt xà ngang
                # Giới hạn trần Y từ (y_ground + goal_height) đến tối đa 4.5 để không chạm Title
                y_miss = random.uniform(y_ground + goal_height + 0.1, 4.5)
            else:
                # Nếu ở hai bên biên thì có thể bay từ mặt đất lên tầm cao vừa phải
                y_miss = random.uniform(y_ground, 4.2)
                
            dot = Dot(point=[x_miss, y_miss, 0], color=RED, radius=0.06)
            all_dots.add(dot)

        # Xáo trộn thứ tự xuất hiện
        all_dots.submobjects = random.sample(all_dots.submobjects, len(all_dots.submobjects))

        # Hiệu ứng "mưa chấm" xuất hiện
        self.play(
            LaggedStart(
                *[FadeIn(d) for d in all_dots],
                lag_ratio=0.008,
                run_time=2.2
            )
        )
        self.wait(0.5)

        # 5. Hiển thị tỷ lệ thực nghiệm (Đặt ở dưới cùng)
        ratio_text = MathTex(
            "P_{thuc \ nghiem} = \\frac{2700}{10000} = 0.27", 
            color=YELLOW
        ).scale(0.8)
        ratio_text.next_to(big_player, DOWN, buff=0.3)
        
        self.play(Write(ratio_text))
        self.wait(1.5)
        self.wait(1)

        # Khởi tạo trước Ma trận Vector X và các thành phần (Chưa hiển thị)
        vector_x = MathTex(
            "X = \\begin{bmatrix} x_1 \\\\ x_2 \\\\ x_3 \\\\ \\vdots \\\\ x_n \\end{bmatrix}",
            color=BLUE
        ).scale(0.9).shift(LEFT * 1.5 + UP * 0.5)

        # Chú thích các biến (labels) tách riêng để xuất hiện lần lượt
        label_x1 = Text("x_1: Distance", font_size=18, color=WHITE)
        label_x2 = Text("x_2: Shot Angle", font_size=18, color=WHITE)
        label_x3 = Text("x_3: Shot Type", font_size=18, color=WHITE)
        label_dots = Text("...", font_size=18, color=WHITE)

        # Sắp xếp các dòng chú thích theo hàng dọc bên phải ma trận X
        labels_group = VGroup(label_x1, label_x2, label_x3, label_dots).arrange(
            DOWN, aligned_edge=LEFT, buff=0.4
        ).next_to(vector_x, RIGHT, buff=0.4)

        # Mũi tên kết nối chĩa từ ma trận sang cầu thủ
        arrows = VGroup(*[
            Arrow(
                start=vector_x.get_right(), # Chĩa từ cạnh phải ma trận X sang
                end=np.array([1.5, -2.5, 0]) + LEFT * 1.2, # Hướng tới rìa trái của ảnh cầu thủ
                buff=0.15, 
                stroke_width=2, 
                color=GRAY_A
            ) for _ in range(2)
        ])

        # STEP 1: Xóa khung thành, bóng, chấm xanh/đỏ và phân số (CHỈ ĐỂ LẠI CẦU THỦ)
        # Lưu ý: "complete_goal" và "all_dots" là từ phân cảnh trước của ông
        self.play(
            FadeOut(complete_goal),
            FadeOut(suppose_text),
            FadeOut(all_dots),
            FadeOut(ratio_text),
            run_time=1.2
        )
        self.wait(0.3)

        # STEP 2: Cầu thủ dịch chuyển sang bên phải tại tọa độ [1.5, -2.5, 0]
        # Đồng thời nhãn tên "GOAT" cũng xuất hiện bám theo ngay dưới chân
        self.play(
            big_player.animate.move_to([1.5, -2.5, 0]),
            run_time=1.2,
            rate_func=smooth
        )
        self.wait(0.5)

        # STEP 3: Ma trận X xuất hiện bên trái màn hình
        self.play(
            Write(vector_x),
            run_time=1.2
        )
        self.wait(0.3)

        # STEP 4: Các lý do x_1 đến x_3 lần lượt xuất hiện, theo sau bởi dấu ba chấm "..."
        self.play(Write(label_x1), run_time=0.6)
        self.play(Write(label_x2), run_time=0.6)
        self.play(Write(label_x3), run_time=0.6)
        self.play(Write(label_dots), run_time=0.4) # Dấu ... xuất hiện cuối cùng dưới x_3
        self.wait(0.5)

        # STEP 5: Cuối cùng, các mũi tên mới chĩa từ ma trận/chú thích về phía cầu thủ
        self.play(
            Create(arrows),
            run_time=1.2
        )
        # Subscene 2: Chuyển đổi từ mô phỏng sút bóng sang biểu diễn hình học thực địa
        self.wait(1)

        # STEP 1: Biến mất toàn bộ vật thể từ phân cảnh trước
        self.play(
            FadeOut(vector_x),
            FadeOut(label_x1), FadeOut(label_x2), FadeOut(label_x3), FadeOut(label_dots),
            FadeOut(arrows),
            FadeOut(big_player),
            run_time=1
        )
        self.wait(0.3)

        # STEP 2: Khung thành hoàn chỉnh xuất hiện
        goal_width = 5.0
        goal_height = 2.5
        y_ground = 1.5 

        post_bottom_left  = np.array([-goal_width/2, y_ground, 0])
        post_bottom_right = np.array([goal_width/2, y_ground, 0])
        goal_top_left     = np.array([-goal_width/2, y_ground + goal_height, 0])
        goal_top_right    = np.array([goal_width/2, y_ground + goal_height, 0])
        goal_center_point = np.array([0, y_ground, 0])

        left_post  = Line(post_bottom_left, goal_top_left, color=WHITE, stroke_width=4)
        right_post = Line(post_bottom_right, goal_top_right, color=WHITE, stroke_width=4)
        crossbar   = Line(goal_top_left, goal_top_right, color=WHITE, stroke_width=4)
        ground_line = Line(np.array([-3.5, y_ground, 0]), np.array([3.5, y_ground, 0]), color=WHITE, stroke_width=2)
        
        complete_goal = VGroup(ground_line, left_post, right_post, crossbar)

        self.play(Create(complete_goal), run_time=1.2)
        self.wait(0.5)

        # STEP 3: TRANSITION - Biến đổi khung thành thành trục Ox và hiện hệ trục Oxy
        axes = Axes(
            x_range=[-3, 3, 1],
            y_range=[0, 5, 1],
            x_length=5.5,
            y_length=4.0,
            axis_config={"color": GRAY, "include_numbers": False}
        ).shift(UP * 1.5)

        self.play(
            ReplacementTransform(complete_goal, axes.x_axis),
            Create(axes.y_axis),
            run_time=1.5
        )

        # Lấy một điểm bất kỳ và NỐI XIÊN xuống trục Ox (Thay vì kẻ vuông góc)
        pt_any = axes.c2p(1.5, 3.5, 0)
        pt_on_ox = axes.c2p(0.5, 0, 0) # Điểm bất kỳ trên trục đại diện cho khung thành

        dot_p = Dot(pt_any, color=YELLOW, radius=0.08)
        lbl_p = MathTex("P(x, y)", font_size=20, color=YELLOW).next_to(pt_any, UR, buff=0.1)
        
        # Đường thẳng nối xiên thể hiện khoảng cách d hình học
        dist_line_axes = Line(pt_any, pt_on_ox, color=BLUE, stroke_width=3)
        lbl_d_axes = MathTex("d", color=BLUE).scale(0.8).next_to(dist_line_axes.get_center(), RIGHT, buff=0.1)

        self.play(FadeIn(dot_p, lbl_p), run_time=0.5)
        self.play(Create(dist_line_axes), Write(lbl_d_axes), run_time=1)

        # Minh họa công thức khoảng cách ngay bên cạnh
        formula_d = MathTex(
            "d = \\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}",
            color=YELLOW
        ).scale(0.8).next_to(axes, DOWN, buff=0.4)

        self.play(Write(formula_d))
        self.wait(2)

        # STEP 4: Trục tọa độ biến mất, phục hồi lại khung thành bình thường từ trục Ox
        rebuild_goal = VGroup(
            Line(np.array([-3.5, y_ground, 0]), np.array([3.5, y_ground, 0]), color=WHITE, stroke_width=2),
            Line(post_bottom_left, goal_top_left, color=WHITE, stroke_width=4),
            Line(post_bottom_right, goal_top_right, color=WHITE, stroke_width=4),
            Line(goal_top_left, goal_top_right, color=WHITE, stroke_width=4)
        )

        self.play(
            FadeOut(dot_p), FadeOut(lbl_p),
            FadeOut(dist_line_axes), FadeOut(lbl_d_axes),
            FadeOut(formula_d),
            FadeOut(axes.y_axis),
            ReplacementTransform(axes.x_axis, rebuild_goal),
            run_time=1.5
        )
        self.wait(0.3)

        # STEP 5: Xuất hiện cầu thủ Díaz, thủ môn Onana (size 2.5) ở gôn, và trái bóng
        onana_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\onana.png"
        onana = ImageMobject(onana_img_path)
        onana.set_width(1.5) 
        onana.move_to([0, y_ground + 0.8, 0]) 

        player_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\baycho.png"
        player_kick = ImageMobject(player_img_path)
        player_kick.set_width(2.5)
        player_kick.move_to([0, -2.5, 0])

        ball_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\ball.png"
        ball = ImageMobject(ball_img_path)
        ball.set_width(0.3)
        ball.move_to(player_kick.get_center() + UP * 0.5 + RIGHT * 0.2)

        self.play(
            FadeIn(onana, shift=DOWN),
            FadeIn(player_kick, shift=UP),
            FadeIn(ball),
            run_time=1.2
        )
        self.wait(0.5)

        # STEP 6: Sút bóng vào góc phải khung thành & Biểu diễn hình học thực địa
        # Định nghĩa góc lưới bên phải làm điểm bóng găm vào gôn
        shot_target = np.array([goal_width/2 - 0.4, y_ground + 0.6, 0]) 

        # Quả bóng bay vào lưới, Onana đổ người hụt sang phải
        self.play(
            ball.animate.move_to(shot_target).scale(0.8),
            onana.animate.shift(RIGHT * 0.6 + DOWN * 0.2), 
            run_time=0.5,
            rate_func=linear
        )
        self.wait(0.3)

        # Nối đường thẳng tính khoảng cách d (từ cầu thủ đến tâm vạch gôn)
        real_line_d = Line(player_kick.get_center(), goal_center_point, color=BLUE, stroke_width=3)
        real_lbl_d = MathTex("d", color=BLUE).scale(0.9).next_to(real_line_d.get_center(), LEFT, buff=0.15)

        self.play(Create(real_line_d), Write(real_lbl_d), run_time=1)

        # BIỂU THỊ GÓC THETA CHỈ VỚI CỘT DỌC BÊN PHẢI (Nơi trái bóng bay tới)
        p_center = player_kick.get_center()
        
        # Đường thứ 1: Hướng bóng bay vào lưới (shot_target)
        ray_shot = Line(p_center, shot_target, color=GRAY_B, stroke_width=2)


        # Tạo vòng cung góc Theta kẹp giữa đường bóng và cột dọc phải
        theta_arc = ArcBetweenPoints(
            start=p_center + 0.6 * normalize(post_bottom_right - p_center),
            end=p_center + 0.6 * normalize(shot_target - p_center),
            stroke_width=2.5,
            color=GREEN
        )
        real_lbl_theta = MathTex("\\theta", color=GREEN).scale(0.9).next_to(theta_arc, UP, buff=0.15)

        self.play(
            Create(ray_shot),
            run_time=0.8
        )
        self.play(
            Create(theta_arc),
            Write(real_lbl_theta),
            run_time=0.8
        )
        self.wait(2)
######################
scene = mainScene()
scene.render()
video_file = Path(config.media_dir) / "videos" / "1280p15" / "mainScene.mp4"
if video_file.exists():
    display(Video(str(video_file), embed=True, width=300))
else:
    found = list(Path(config.media_dir).glob("**/mainScene.mp4"))
    if found:
        display(Video(str(found[0]), embed=True, width=300))
    else:
        print("Render xong nhưng không tìm thấy file mp4 để hiển thị.")